# Coveo Dataset EDA
## Goal: understand session structure before building anything

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path

%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
# place Coveo browsing_train.csv here
data_path = Path('../../data/raw/browsing_train.csv')

df = pd.read_csv(data_path)
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head(10)

In [ ]:
# Session length analysis
session_lengths = df.groupby('session_id').size()

print(session_lengths.describe())

fig, ax = plt.subplots(figsize=(10, 4))
session_lengths.clip(upper=50).hist(bins=50, ax=ax)
ax.set_xlabel('Session length (events)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Session Lengths (capped at 50)')
plt.tight_layout()
plt.show()

In [ ]:
# Event type breakdown
event_counts = df['event_type'].value_counts()
print(event_counts)

fig, ax = plt.subplots(figsize=(8, 4))
event_counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Event type')
ax.set_ylabel('Count')
ax.set_title('Event Type Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Query frequency — what fraction of sessions contain at least one search query
sessions_with_query = df[df['event_type'] == 'query']['session_id'].nunique()
total_sessions = df['session_id'].nunique()
query_fraction = sessions_with_query / total_sessions

print(f'Total sessions:              {total_sessions:,}')
print(f'Sessions with >= 1 query:    {sessions_with_query:,}')
print(f'Fraction with search query:  {query_fraction:.2%}')

In [ ]:
# Exploratory query similarity analysis
# For sessions with 2+ queries, compute cosine similarity between consecutive
# query embeddings using sentence-transformers all-MiniLM-L6-v2.
# Exploratory only:
# Do not assume a universal intent-shift threshold.
# Analyze the similarity distribution and validate the definition later.

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract sessions with 2+ queries
query_events = df[df['event_type'] == 'query'].copy()
session_query_counts = query_events.groupby('session_id').size()
multi_query_sessions = session_query_counts[session_query_counts >= 2].index

print(f'Sessions with 2+ queries: {len(multi_query_sessions):,}')

# Compute consecutive query similarities (sample up to 5000 sessions for speed)
sample_sessions = multi_query_sessions[:5000]
all_similarities = []

for sid in sample_sessions:
    queries = query_events[query_events['session_id'] == sid]['query'].dropna().tolist()
    if len(queries) < 2:
        continue
    embeddings = model.encode(queries, show_progress_bar=False)
    for i in range(len(embeddings) - 1):
        sim = cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0]
        all_similarities.append(sim)

all_similarities = np.array(all_similarities)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_similarities, bins=50)
ax.set_xlabel('Cosine similarity between consecutive queries')
ax.set_ylabel('Count')
ax.set_title('Exploratory Consecutive Query Similarity Distribution')
plt.tight_layout()
plt.show()

print('Similarity quantiles:', np.quantile(all_similarities, [0, 0.25, 0.5, 0.75, 1]))

In [ ]:
# Item interaction distribution — cold start analysis
item_interactions = df[df['event_type'] == 'click']['product_sku_hash'].value_counts()

cold_start_threshold = 5
cold_start_items = (item_interactions < cold_start_threshold).sum()
total_items = len(item_interactions)

print(f'Total unique items:                          {total_items:,}')
print(f'Items with fewer than {cold_start_threshold} interactions:    {cold_start_items:,}')
print(f'Cold-start item percentage:                  {cold_start_items / total_items:.2%}')

## Findings

- 
- 
- 
- 
- 